# Activity: SVM scores, signed distance, and the kernel trick

**Purpose:** work out what an SVM decision function computes and how a kernel replaces an explicit feature-space dot product. This is an in-class learning activity: discuss predictions and calculations with a neighbor, edit the `TODO` lines, and use the checks for feedback. Nothing here is timed or graded.

By the end, you should be able to:

1. Distinguish the raw decision score $f(x)=w^\top x+b$ from signed perpendicular distance $f(x)/\lVert w\rVert$.
2. Explain why the $f(x)=\pm1$ margin lines are a choice of score scale, while their distance from $f(x)=0$ is $1/\lVert w\rVert$ for a canonically scaled SVM.
3. Calculate a polynomial kernel both from a feature map and directly from the original inputs.
4. Reconstruct a kernel SVM decision score from support-vector contributions.

We use binary labels $-1$ and $+1$. The positive side of a boundary is the side where $f(x)>0$. The examples connect to [the SVM lecture slides](../slides-pdf/12-svm.pdf); this activity focuses on the geometry and kernel calculation rather than tuning $C$ or $\gamma$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

np.set_printoptions(precision=3, suppress=True)

## 1. From a score to a signed distance

Take $w=(3,4)$ and $b=-5$. The boundary is $3x_1+4x_2-5=0$. Let $x=(2,1)$ and choose the boundary point $x_0=(-1,2)$.

**Before running code:**

1. Verify that $x_0$ lies on the boundary.
2. Compute $f(x)=w^\top x+b$ and $\lVert w\rVert$. What do the sign and magnitude of $f(x)$ mean?
3. Project $x-x_0$ onto the **unit** normal $w/\lVert w\rVert$. Show why

   $$d(x)=\frac{(x-x_0)^\top w}{\lVert w\rVert}
   =\frac{w^\top x+b}{\lVert w\rVert}.$$

4. The closest point on the boundary is $x_{\perp}=x-d(x)w/\lVert w\rVert$. Find it by hand.

Fill the four `TODO` values. `score` is in score units; `signed_distance` is in the original feature units because $w$ is normalized.

In [ ]:
w = np.array([3.0, 4.0])
b = -5.0
x = np.array([2.0, 1.0])
x0 = np.array([-1.0, 2.0])

score = None             # TODO: w @ x + b
normal_length = None     # TODO: Euclidean length of w
signed_distance = None   # TODO: score divided by normal_length
foot = None              # TODO: x minus signed_distance times the unit normal

assert all(v is not None for v in (score, normal_length, signed_distance, foot)),     "Replace all four None values after doing the calculation by hand."
assert np.isclose(w @ x0 + b, 0), "The reference point must be on the boundary."
assert np.isclose(w @ foot + b, 0), "The projected point should lie on the boundary."
assert np.isclose(np.linalg.norm(x - foot), abs(signed_distance)),     "A perpendicular projection has length equal to the absolute distance."
print(f"score = {score:.3f}; signed distance = {signed_distance:.3f}")
print("closest boundary point =", foot)

In [ ]:
# The dashed segment should meet the boundary at a right angle.
x1_line = np.linspace(-1.5, 3.0, 200)
x2_line = -(w[0] * x1_line + b) / w[1]
fig, ax = plt.subplots(figsize=(6, 5))
ax.plot(x1_line, x2_line, color="black", label=r"$f(x)=0$")
ax.plot(*x, "o", color="tab:blue", markersize=9, label=r"$x$")
ax.plot(*x0, "o", color="tab:gray", label=r"$x_0$")
ax.plot(*foot, "o", color="tab:orange", label=r"$x_\perp$")
ax.plot([x[0], foot[0]], [x[1], foot[1]], "--", color="tab:orange")
ax.set(xlim=(-1.5, 3.0), ylim=(-0.7, 3.2), xlabel=r"$x_1$", ylabel=r"$x_2$",
       title="Projecting onto the SVM boundary")
ax.set_aspect("equal")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

### Change the score scale, keep the geometry

Multiply **both** $w$ and $b$ by a positive number $t=3$. Predict the new raw score and signed distance for $x$ before running the cell. Why does the boundary stay put? What would a negative multiplier do to the sign convention?

> **Your prediction and explanation:**

In [ ]:
t = 3.0
w_scaled = t * w
b_scaled = t * b
scaled_score = None      # TODO: score at x using w_scaled and b_scaled
scaled_distance = None   # TODO: signed geometric distance using w_scaled

assert scaled_score is not None and scaled_distance is not None,     "Compute both values before comparing them."
assert np.isclose(scaled_score, t * score), "Check how a positive rescaling affects the score."
assert np.isclose(scaled_distance, signed_distance),     "Check the norm in the denominator of the distance formula."
print(f"original score {score:.3f}, scaled score {scaled_score:.3f}")
print(f"original distance {signed_distance:.3f}, scaled distance {scaled_distance:.3f}")

## 2. Read an actual linear SVM decision function

Our one-feature data have negative points at $-3,-2$ and positive points at $2,3$. Fit a nearly hard-margin linear SVM. For this dataset the fit is simple enough to inspect by hand. `SVC.decision_function` returns $f(x)$, **not** $f(x)/\lVert w\rVert$.

**Before running:** Where would you place the boundary? Which observations should touch the margin? Estimate their geometric distance from the boundary.

> **Your prediction:**

In [ ]:
X_linear = np.array([[-3.0], [-2.0], [2.0], [3.0]])
y_linear = np.array([-1, -1, 1, 1])
linear_svm = SVC(kernel="linear", C=10_000).fit(X_linear, y_linear)
w_fit = linear_svm.coef_.ravel()
b_fit = float(linear_svm.intercept_[0])

print("w =", w_fit, "b =", b_fit)
print("support points =", X_linear[linear_svm.support_].ravel())
print("decision_function at training points =", linear_svm.decision_function(X_linear))

For each training point, compute $f(x_i)$, its **signed** distance $d_i=f(x_i)/\lVert w\rVert$, and its label-adjusted score $y_i f(x_i)$. For the four points, make a small table on paper first. Add a row for $x=0$ (which has no class label). Which two points attain $y_i f(x_i)=1$?

In a separable, canonically scaled SVM, the nearest training points have $y_i f(x_i)=1$. Therefore the margin **on one side** has width $1/\lVert w\rVert$; the whole street between $f=-1$ and $f=+1$ has width $2/\lVert w\rVert$.

> **Your table and interpretation:**

In [ ]:
manual_scores = None       # TODO: X_linear @ w_fit + b_fit
manual_distances = None    # TODO: divide manual_scores by ||w_fit||
label_scores = None        # TODO: y_linear * manual_scores
one_side_margin = None     # TODO: 1 / ||w_fit||

assert all(v is not None for v in
           (manual_scores, manual_distances, label_scores, one_side_margin)),     "Fill in all four quantities after making the table."
np.testing.assert_allclose(manual_scores, linear_svm.decision_function(X_linear), atol=1e-8)
np.testing.assert_allclose(manual_distances * np.linalg.norm(w_fit), manual_scores)
assert np.all(label_scores >= 1 - 1e-3), "Check the label-adjusted scores."
assert np.isclose(one_side_margin * np.linalg.norm(w_fit), 1)
print("  x   label   score   signed distance   y*score")
for xi, yi, fi, di, mi in zip(X_linear.ravel(), y_linear,
                              manual_scores, manual_distances, label_scores):
    print(f"{xi:4.0f} {yi:6d} {fi:8.3f} {di:17.3f} {mi:9.3f}")
print("one-side geometric margin =", one_side_margin)

In [ ]:
grid_linear = np.linspace(-3.5, 3.5, 200)
f_grid = linear_svm.decision_function(grid_linear[:, None])
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(grid_linear, f_grid, color="tab:blue", label=r"$f(x)$")
for level, style in [(-1, "--"), (0, "-"), (1, "--")]:
    ax.axhline(level, color="black", linestyle=style, alpha=0.5)
ax.scatter(X_linear.ravel(), manual_scores, c=y_linear, cmap="coolwarm",
           vmin=-1, vmax=1, edgecolor="black", zorder=3, label="training points")
ax.set(xlabel="original feature x", ylabel="decision score f(x)",
       title="Scores of a canonically scaled linear SVM")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

**Discuss:** Here $\lVert w\rVert$ is not $1$. Why would it be wrong to label the dashed $f=\pm1$ lines as “one unit from the boundary”? If we multiplied this fitted $(w,b)$ by $3$, would the new $f=\pm1$ lines still pass through the same training points?

> **Your explanation:**

## 3. Make a nonlinear problem linear in a feature space

The labels below alternate as we move from the center to the ends of a one-dimensional line. No single threshold in the original $x$ coordinate separates them. But $x^2$ separates the outer points from the inner points: for example, $g(x)=x^2-2.5$ has the right **signs** for all five observations. This $g$ is a hand-chosen separator, not yet an SVM fit.

| $x$ | $-2$ | $-1$ | $0$ | $1$ | $2$ |
|---:|---:|---:|---:|---:|---:|
| $y$ | $+1$ | $-1$ | $-1$ | $-1$ | $+1$ |

**Sketch:** Put the five points on a number line. Then plot their $(x,x^2)$ coordinates. Explain why a horizontal line can separate the classes in the new picture.

> **Your sketch/explanation:**

In [ ]:
x_train = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
y_train = np.array([1, -1, -1, -1, 1])
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for xi, yi in zip(x_train, y_train):
    axes[0].scatter(xi, 0, color="tab:blue" if yi == 1 else "tab:orange", s=80)
    axes[1].scatter(xi, xi**2, color="tab:blue" if yi == 1 else "tab:orange", s=80)
axes[1].axhline(2.5, color="black", linestyle="--", label=r"$x^2=2.5$")
axes[0].set(xlabel="x", yticks=[], title="Original one-dimensional space")
axes[1].set(xlabel="x", ylabel=r"$x^2$", title="Explicit quadratic feature")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
plt.show()

### Calculate the feature-space dot product two ways

Use the feature map $\phi(x)=(1,\sqrt{2}x,x^2)$. For two original values $a$ and $c$, expand the dot product:

$$\phi(a)^\top\phi(c)
=1+2ac+a^2c^2
=(1+ac)^2
=K(a,c).$$

The last expression uses only $a$ and $c$: it is the degree-2 polynomial kernel with `gamma=1` and `coef0=1`. For $a=1$ and $c=2$, compute $\phi(a)$, $\phi(c)$, their dot product, and $K(a,c)$ **by hand**. Circle the cross-term that requires the $\sqrt{2}$ in $\phi$.

> **Your calculation:**

Complete both functions. Each input to `phi` is a one-dimensional array of original values; `poly2` returns a matrix with one row per value in `a` and one column per value in `c`.

In [ ]:
def phi(x):
    x = np.asarray(x, dtype=float).reshape(-1)
    return None  # TODO: columns 1, sqrt(2)*x, x**2; shape (len(x), 3)


def poly2(a, c):
    a = np.asarray(a, dtype=float).reshape(-1)
    c = np.asarray(c, dtype=float).reshape(-1)
    return None  # TODO: (1 + a[:, None] * c[None, :])**2


Phi = phi(x_train)
pair_kernel = poly2([1.0], [2.0])
assert Phi is not None and pair_kernel is not None, "Implement both functions."
assert Phi.shape == (len(x_train), 3), "phi should produce three features per point."
assert pair_kernel.shape == (1, 1), "poly2 should return a pairwise matrix."
np.testing.assert_allclose(phi([1.0]) @ phi([2.0]).T, pair_kernel)
print("phi(1) =", phi([1.0])[0], "; phi(2) =", phi([2.0])[0])
print("feature dot product =", float((phi([1.0]) @ phi([2.0]).T)[0, 0]))
print("kernel value =", pair_kernel[0, 0])

### Build the pairwise similarity table

Before coding, calculate $K(-2,1)$, $K(-2,2)$, and $K(0,2)$ by hand. A larger polynomial-kernel dot product does **not** necessarily mean two points are closer in original $x$ space. Which of the first two pairs illustrates that?

> **Your three calculations and interpretation:**

The training Gram matrix has entry $G_{ij}=K(x_i,x_j)$. Compute it without calling `phi`, then check it against explicit feature-space dot products.

In [ ]:
K_train = None  # TODO: use poly2 on all pairs from x_train; shape (5, 5)

assert K_train is not None, "Compute the pairwise kernel matrix first."
assert K_train.shape == (len(x_train), len(x_train))
np.testing.assert_allclose(K_train, Phi @ Phi.T, atol=1e-12)
print("rows and columns correspond to x =", x_train)
print(K_train)

## 4. Fit the same SVM with and without explicit features

The first model fits a linear SVM to the three columns of $\phi(x)$. The second model receives only the Gram matrix. `kernel="precomputed"` tells `SVC` that every entry is already a kernel dot product. Both models use the same $C$ and the same feature-space geometry. Their numerical scores may differ slightly because the two solvers stop at finite tolerance.

**Predict:** What shape must the matrix of kernel values for 301 new grid points have? Why does it need columns for **training** observations?

> **Your prediction:**

In [ ]:
C = 100.0
explicit_svm = SVC(kernel="linear", C=C).fit(Phi, y_train)
kernel_svm = SVC(kernel="precomputed", C=C).fit(K_train, y_train)

x_grid = np.linspace(-2.5, 2.5, 301)
K_grid = None  # TODO: kernel values between x_grid (rows) and x_train (columns)

assert K_grid is not None, "Compute the grid-to-training kernel matrix."
assert K_grid.shape == (len(x_grid), len(x_train))
explicit_grid_scores = explicit_svm.decision_function(phi(x_grid))
kernel_grid_scores = kernel_svm.decision_function(K_grid)
np.testing.assert_allclose(explicit_grid_scores, kernel_grid_scores, atol=1e-3)
print("largest score difference =", np.max(np.abs(explicit_grid_scores - kernel_grid_scores)))
print("support-vector locations =", x_train[kernel_svm.support_])

### Where the dual weights come from

The soft-margin SVM can be written in two equivalent ways. The formulation based on a separating hyperplane chooses its weights $w_\phi$ and intercept $b$; the **dual** formulation instead chooses one coefficient $\alpha_i$ for each training observation. It solves

$$\max_{\boldsymbol{\alpha}}\left\{\sum_i\alpha_i-\frac12\sum_{i,j}\alpha_i\alpha_j y_i y_j K(x_i,x_j)\right\}$$

subject to $0\leq\alpha_i\leq C$ and $\sum_i\alpha_i y_i=0$, where $y_i\in\{-1,+1\}$ and $C$ is the penalty used above. At the solution, $w_\phi=\sum_i\alpha_i y_i\phi(x_i)$. Thus the dual uses feature-space dot products $K(x_i,x_j)=\phi(x_i)^\top\phi(x_j)$, and observations with $\alpha_i>0$ are support vectors. We will use this result without deriving the optimization steps.

### Rebuild one kernel prediction from support vectors

For a binary kernel SVM, the fitted decision score is

$$f(x_*)=\sum_{i\in\mathrm{SV}}(\alpha_i y_i)K(x_i,x_*)+b.$$

In scikit-learn, `kernel_svm.dual_coef_[0]` stores the **signed** weights $\alpha_i y_i$ in the same order as `kernel_svm.support_`. `kernel_svm.intercept_[0]` is $b$. The model keeps only support-vector terms because the other $\alpha_i$ are zero.

Take $x_*=1.5$. Inspect the printed support-vector locations and weights. Compute at least one $K(x_i,1.5)$ by hand, then complete the code to add the contributions. Notice that the weighted sum is a **score**; it is not a probability.

> **One hand-calculated kernel value and your predicted score sign:**

In [ ]:
x_star = 1.5
sv_indices = kernel_svm.support_
sv_x = x_train[sv_indices]
signed_weights = kernel_svm.dual_coef_[0]
print("support x:", sv_x)
print("signed weights:", signed_weights)
print("intercept:", kernel_svm.intercept_[0])

k_star_sv = None      # TODO: kernel values from x_star to each support x; shape (n_support,)
manual_score = None    # TODO: signed_weights @ k_star_sv + intercept

assert k_star_sv is not None and manual_score is not None,     "Calculate the support-vector kernel values and add their signed contributions."
assert np.shape(k_star_sv) == signed_weights.shape
library_score = kernel_svm.decision_function(poly2([x_star], x_train))[0]
np.testing.assert_allclose(manual_score, library_score, atol=1e-8)
print("support contributions:", signed_weights * k_star_sv)
print(f"manual score = {manual_score:.4f}; SVC score = {library_score:.4f}")
print("predicted label =", 1 if manual_score > 0 else -1)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_grid, explicit_grid_scores, label="explicit quadratic features", color="tab:blue")
ax.plot(x_grid, kernel_grid_scores, "--", label="polynomial kernel", color="tab:orange")
ax.axhline(0, color="black", alpha=0.6)
ax.scatter(x_train, np.zeros_like(x_train), c=y_train, cmap="coolwarm",
           vmin=-1, vmax=1, edgecolor="black", zorder=3)
ax.scatter([x_star], [manual_score], marker="*", s=170, color="tab:green",
           label=r"$x_*=1.5$")
ax.set(xlabel="original feature x", ylabel="decision score f(x)",
       title="The same feature-space separator, evaluated two ways")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

### Where does signed distance go after the kernel trick?

The quadratic model still has a hyperplane in $\phi$-space. Its signed distance to that **feature-space hyperplane** is $f(x)/\lVert w_\phi\rVert$. The dual coefficients let us compute its normal length without building $w_\phi$:

$$\lVert w_\phi\rVert^2
=\sum_{i,j\in\mathrm{SV}}(\alpha_i y_i)(\alpha_j y_j)K(x_i,x_j).$$

Compute this norm and the feature-space signed distance for $x_*$. Compare with the explicit model's coefficient norm. **Do not interpret** this feature-space distance as the shortest distance, in original $x$ units, to the curved decision boundary. The feature map changes geometry.

> **Why are score, feature-space distance, and original-space distance different quantities?**

In [ ]:
K_sv = K_train[np.ix_(sv_indices, sv_indices)]
feature_normal_length = None  # TODO: sqrt(signed_weights @ K_sv @ signed_weights)
feature_distance = None       # TODO: manual_score / feature_normal_length

assert feature_normal_length is not None and feature_distance is not None,     "Compute the feature-space normal length and distance."
np.testing.assert_allclose(feature_normal_length,
                           np.linalg.norm(explicit_svm.coef_), atol=1e-3)
np.testing.assert_allclose(feature_distance * feature_normal_length, manual_score)
print("feature-space normal length =", feature_normal_length)
print("feature-space signed distance at x* =", feature_distance)

## 5. Explain the connection

Discuss and write brief answers in your own words.

1. What does `decision_function(x)` return for the linear model? What extra operation turns it into signed geometric distance in the original feature space?
2. If $(w,b)$ are multiplied by a positive constant, which of **boundary**, **score**, and **signed distance** change?
3. For $\phi(x)=(1,\sqrt{2}x,x^2)$, what calculation does $K(a,c)=(1+ac)^2$ save? Which input coordinates does the kernel need?
4. Why can `kernel_svm` predict without ever constructing the three columns of `phi(x)`? What role do the support vectors play?
5. What is the geometric meaning of `feature_distance` in the previous cell? Why is it not an original-space distance to the boundary?

> **Your answers:**